To use the AENets model please visit and follow the instructions in the original repository

https://github.com/ZhangYuanhan-AI/CelebA-Spoof/tree/master/intra_dataset_code

# Import Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
!cp -r "/content/gdrive/MyDrive/BiometricsModels/AENet/assets/" .

In [ ]:
import time
import sys
import tqdm
import logging

import cv2
import numpy as np

In [ ]:
import glob
import pandas as pd

In [ ]:
logging.basicConfig(level=logging.INFO)

In [ ]:
from assets.client import get_image, verify_output
from assets.tsn_predict import TSNPredictor as CelebASpoofDetector

# Load Dataset

In [ ]:
!cp "/content/gdrive/MyDrive/DATABASES/MSU_MFSD/face/msu_mfsd_faces.zip" .
!unzip -qq msu_mfsd_faces.zip

In [ ]:
!rm -r msu_mfsd_faces.zip

## Prepare Data to Predict

In [ ]:
np.random.seed(42)
df = pd.read_csv('/content/gdrive/MyDrive/DATABASES/MSU_MFSD/face/msu_mfsd_faces_intradataset.csv', index_col=False)
df.sample(3)

,frame,scene,client,label,data_type,full_path
28466,frame_218.jpg,attack_client005_laptop_SD_printed_photo_scene01,client005,attack,train,faces/train/attack/attack_client005_laptop_SD_...
5533,frame_105.jpg,attack_client051_laptop_SD_ipad_video_scene01,client051,attack,test,faces/test/attack/attack_client051_laptop_SD_i...
9252,frame_118.jpg,attack_client001_android_SD_iphone_video_scene01,client001,attack,test,faces/test/attack/attack_client001_android_SD_...


## Prepare generator data

In [ ]:
column_path = 'full_path'

def batch_image_generator(df, batch_size=16, process_image_func=None):
    total_images = len(df)
    num_batches = int(np.ceil(total_images/BATCH))

    for batch_idx in range(num_batches):

        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, total_images)

        img_batch_list = []
        for idx in range(start_idx, end_idx):
            image_path = df.iloc[idx][column_path]

            try:

                img = cv2.imread(image_path)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (224, 224))

                img_batch_list.append(img)

            except Exception as e:
                print(f"Unexpected error {image_path}: {e}")

        yield np.array(img_batch_list, dtype=object).astype('uint8')

In [ ]:
BATCH = 64
df_process = df.copy()

image_generator_test = batch_image_generator(df_process, batch_size=BATCH)

total_images = len(df_process)
num_batches = (total_images + BATCH - 1) // BATCH
total_images, num_batches

(34027, 532)

# Pretrained Model

## Load model

In [ ]:
detector_pretreined = CelebASpoofDetector()

/content/assets/tsn_predict.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('./assets/ckpt_iter.pth.tar')


## Predict

In [ ]:
predicts = []
for batch in tqdm.tqdm(image_generator_test, total=num_batches):
    try:
        predicts.extend(detector_pretreined.predict(batch))
    except Exception as e:
        print(f"Unexpected error: {e}")
        predicts.extend([None] * len(batch))
len(predicts)

100%|██████████| 532/532 [05:16<00:00,  1.68it/s]


34027

In [ ]:
df_process['label_predict_proba'] = np.array(predicts).tolist()

In [ ]:
df_process.head(3)

,frame,scene,client,label,data_type,full_path,label_predict_proba
0,frame_125.jpg,attack_client042_laptop_SD_ipad_video_scene01,client042,attack,test,faces/test/attack/attack_client042_laptop_SD_i...,"[0.630559504032135, 0.3694404661655426]"
1,frame_4.jpg,attack_client042_laptop_SD_ipad_video_scene01,client042,attack,test,faces/test/attack/attack_client042_laptop_SD_i...,"[0.8646532297134399, 0.13534674048423767]"
2,frame_46.jpg,attack_client042_laptop_SD_ipad_video_scene01,client042,attack,test,faces/test/attack/attack_client042_laptop_SD_i...,"[0.9211543202400208, 0.07884563505649567]"


In [ ]:
path_dir = '/content/gdrive/MyDrive/csv_results/'

In [ ]:
df_process.to_csv(f'{path_dir}AENet_prot3_trCeAS_tsMSU.csv', index=False)

END